# Random forest on consensus labels
Run after notebook 04. Uses the same training sections, optical channels, held-out manifests and MRI references. Scores measure agreement with MRI.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from brainseg.benchmark import evaluate
from brainseg.compare import compare
from brainseg.methods import unet
from brainseg.methods.classical import segment
from brainseg.mri_labels import consensus

DATA, OUT = Path("../data"), Path("../outputs")
BENCH, CACHE, MODELS = DATA / "benchmark_mri", DATA / "unet", OUT / "unet"
SUBJECTS = {"vlad": 0, "moe": 1}  # slide-deck axis that indexes sections
OTHER = {"vlad": "moe", "moe": "vlad"}
SETTINGS = {"iterations": 4000, "patch": 256, "batch": 16, "lr": 1e-3, "width": 16, "depth": 4, "seed": 0}
CACHE.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print("device:", unet.device())

In [ ]:
def load_brain(s):
    paths = {k: CACHE / f"{s}_{k}.npy" for k in ("x", "otsu", "mri", "labels", "index")}
    if not all(p.exists() for p in paths.values()):
        axis = SUBJECTS[s]
        mri = np.moveaxis(np.asarray(nib.load(OUT / f"{s}_labels_psoct.nii.gz").dataobj), axis, 0)
        x = np.stack([np.moveaxis(np.asarray(nib.load(DATA / s / "warp" / f"{c}_slide_deck.nii.gz").dataobj, dtype=np.float32), axis, 0)
                      for c in ("Ret", "Ref")], -1)
        index = np.flatnonzero(np.isin(mri, (1, 2, 3)).any((1, 2)))
        otsu = np.stack([segment(x[i], "otsu", retardance=0, reflectivity=1, pixel_size=0.055) for i in index])
        for k, v in {"x": x[index], "otsu": otsu, "mri": mri[index], "labels": consensus(mri[index], otsu), "index": index}.items():
            np.save(paths[k], v)
    return {k: np.load(p, mmap_mode="r") for k, p in paths.items()}

brains = {s: load_brain(s) for s in SUBJECTS}
for s, b in brains.items():
    mri, labels = np.asarray(b["mri"]), np.asarray(b["labels"])
    kept = {name: (labels == c).sum() / max((mri == c).sum(), 1) for c, name in enumerate(("background", "gray", "white", "fluid"))}
    print(f"{s}: {len(b['index'])} sections; consensus keeps " + ", ".join(f"{k} {v:.0%}" for k, v in kept.items()) + " of MRI labels")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from brainseg.methods.baselines import features
from brainseg.preprocess import unimaged

def fit_forest(brain, seed=0, budget=200000):
    rng = np.random.default_rng(seed)
    def mask(a, y):
        return (y < 4) & ~unimaged(a)
    counts = [mask(a, y).sum() for a, y in zip(brain["x"], brain["labels"])]
    total = sum(counts)
    if not total:
        raise ValueError("No eligible training pixels")
    # Sample uniformly across the whole brain, then extract features one section at a time.
    ranks = np.sort(rng.choice(total, min(budget, total), replace=False))
    xs, ys, start = [], [], 0
    for a, y, count in zip(brain["x"], brain["labels"], counts):
        end = start + count
        local = ranks[np.searchsorted(ranks, start):np.searchsorted(ranks, end)] - start
        if len(local):
            pixels = np.flatnonzero(mask(a, y))[local]
            f = features(a, "rf")
            xs.append(f.reshape(-1, f.shape[-1])[pixels])
            ys.append(y.ravel()[pixels])
        start = end
    return RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                  random_state=seed, n_jobs=-1).fit(np.concatenate(xs), np.concatenate(ys))

def predict_forest(model, a):
    valid = ~unimaged(a)
    out = np.zeros(valid.shape, dtype=np.uint8)
    if valid.any():
        out[valid] = model.predict(features(a, "rf")[valid])
    return out

In [ ]:
from brainseg.io import read_manifest

runs = []
for train, test in OTHER.items():
    model = fit_forest(brains[train])
    for suffix in ("", "_band"):
        manifest = BENCH / f"manifest{suffix}_{test}.csv"
        rows = read_manifest(manifest)
        if not rows or any(r["sample_id"] != test for r in rows):
            raise ValueError(f"Expected only {test} in {manifest}")
        for name in ("rf", "rf_masked"):
            def predictor(a):
                pred = predict_forest(model, a)
                if name == "rf_masked":
                    otsu = segment(a, "otsu", retardance=0, reflectivity=1, pixel_size=0.055)
                    split = np.isin(otsu, (1, 2)) & np.isin(pred, (1, 2))
                    return np.where(split, pred, otsu)
                return pred
            run = OUT / "mri_benchmark" / f"{name}_from_{train}{suffix}"
            evaluate(manifest, "test", run, name, predictor, train_sample_ids=[train])
            runs.append(run)
        for name in ("unet", "unet_masked"):
            runs.append(OUT / "mri_benchmark" / f"{name}_from_{train}{suffix}")
runs.extend(OUT / "mri_benchmark" / f"otsu{s}" for s in ("", "_band"))
table = compare(runs, OUT / "mri_benchmark" / "table_rf_unet.csv")
for r in table:
    if r["metric"] == "dice" and r["class"] in ("background", "gray_matter", "white_matter"):
        print(r["run"], r["sample_id"], r["class"], r["mean"])